# LLM Experiment: Qwen 3B Chat + Layer Activations

This notebook walks through:
1. Logging into Hugging Face with a token entered at runtime.
2. Loading a Qwen 3B instruct model.
3. Running an interactive chat loop.
4. Printing per-layer activation statistics for each user prompt.

> If the default model ID changes on Hugging Face, update `MODEL_ID` in the load cell.

In [ ]:
# If needed, uncomment and run this once:
# %pip install -q torch transformers huggingface_hub

import getpass
import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Step 1: Log in to Hugging Face with a token entered securely at runtime.
hf_token = getpass.getpass("Enter your Hugging Face token: ")
login(token=hf_token)
print("Hugging Face login successful.")

In [ ]:
# Step 2: Load Qwen 4B Instruct model.
# Update this if the model repo name changes.
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"Using device={device}, dtype={dtype}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None,
)

if device in {"cpu", "mps"}:
    model = model.to(device)

model.eval()
print("Model loaded.")

In [ ]:
# Step 3: Activation inspection for each user prompt.
@torch.no_grad()
def print_layer_activations_for_prompt(user_prompt: str, system_prompt: str = "You are a helpful assistant."):
    chat = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    prompt_text = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    outputs = model(**inputs, output_hidden_states=True, use_cache=False)

    hidden_states = outputs.hidden_states  # tuple: (embeddings, layer1, layer2, ...)
    last_token_norms = []

    for layer_idx, layer_tensor in enumerate(hidden_states):
        # layer_tensor shape: [batch, seq_len, hidden_dim]
        vec = layer_tensor[0, -1, :].float()
        l2_norm = torch.linalg.vector_norm(vec).item()
        mean_abs = vec.abs().mean().item()
        max_abs = vec.abs().max().item()
        last_token_norms.append((layer_idx, l2_norm, mean_abs, max_abs))

    print("\nActivation summary (last token of formatted prompt):")
    print(f"User prompt: {user_prompt}")
    print("layer\tl2_norm\tmean_abs\tmax_abs")
    for layer_idx, l2_norm, mean_abs, max_abs in last_token_norms:
        print(f"{layer_idx}\t{l2_norm:.4f}\t{mean_abs:.4f}\t{max_abs:.4f}")

In [ ]:
# Step 4: Chat loop that prints layer activations for each user input.
@torch.no_grad()
def generate_assistant_reply(chat_history, max_new_tokens: int = 256):
    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def chat_with_activation_prints(system_prompt: str = "You are a helpful assistant."):
    print("Type 'exit' to stop chat.\n")
    chat_history = [{"role": "system", "content": system_prompt}]

    while True:
        user_text = input("You: ").strip()
        if not user_text:
            continue
        if user_text.lower() in {"exit", "quit"}:
            print("Chat ended.")
            break

        # Print per-layer activations for this user prompt.
        print_layer_activations_for_prompt(user_text, system_prompt=system_prompt)

        chat_history.append({"role": "user", "content": user_text})
        reply = generate_assistant_reply(chat_history)
        chat_history.append({"role": "assistant", "content": reply})

        print(f"\nAssistant: {reply}\n")

In [ ]:
# Step 5: Start chatting.
chat_with_activation_prints()